# Homework 2: BERT on AWS

In [22]:
%pip install boto3

Note: you may need to restart the kernel to use updated packages.


In [23]:
!pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


In [24]:
!pip install --no-cache-dir transformers

In [25]:
import requests
import boto3

# Check if dataset is already in S3 bucket 
def check_bucket_for_dataset(bucket_name, file_key):
    try: 
        # already in bucket
        boto3.client('s3').head_object(Bucket=bucket_name, Key=file_key)
        return True
    except: 
        # not in bucket
        return False

# Download dataset into S3 bucket
def download_dataset(bucket_name, dataset_name, dataset_url, file_key):
    response = requests.get(dataset_url)
    with open(dataset_name, 'wb') as f:
        f.write(response.content)

    boto3.client('s3').upload_file(dataset_name, bucket_name, file_key)
    

# Run both functions 
def attempt_download(bucket_name, file_key, dataset_name, dataset_url):
    if not check_bucket_for_dataset(bucket_name, file_key):
        download_dataset(bucket_name, dataset_name, dataset_url, file_key)
        print("Downloaded dataset to S3 bucket.")
    else: 
        print("Already downloaded. Nothing to do.")

attempt_download("dongjoo-26spring", "data/ml-1m.zip", "ml-1m.zip", "https://files.grouplens.org/datasets/movielens/ml-1m.zip")

Already downloaded. Nothing to do.


## Task 2: Create Embeddings

In [26]:
import zipfile

# Get dataset from S3 bucket
def extract_dataset_from_s3(bucket_name, s3_key, local_zip, extract_to):
    boto3.client('s3').download_file(bucket_name, s3_key, local_zip)
    print("Retrieving file from S3 bucket")
    with zipfile.ZipFile(local_zip, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"Done! Files are now in the '{extract_to}' folder on your instance.")

extract_dataset_from_s3("dongjoo-26spring", "data/ml-1m.zip", "ml-1m.zip", "data/")

Retrieving file from S3 bucket
Done! Files are now in the 'data/' folder on your instance.


In [27]:
import pandas as pd
import re

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Initialize tokenizer and model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

# Helper to extract year from movie title
def get_year_and_fix_title(row):
    original_title = row['title']

    is_year = re.search(r'\((\d{4})\)', original_title)
    year = int(is_year.group(1)) 

    clean_title = re.sub(r'\s\(\d{4}\)$', '', original_title)
    
    return pd.Series([clean_title, year])

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
# Fetch movie dataset and filter, creating new columns "clean_title" and "year"
def filter_movies(file_path):
    columns = ['movie_id', 'title', 'genres']
    movies_df = pd.read_csv(
        file_path, 
        sep='::',  
        engine='python', 
        encoding='latin-1',
        names=['movie_id', 'title', 'genres']
    )

    movies_df[['clean_title', 'year']] = movies_df.apply(get_year_and_fix_title, axis=1)
    movies_filtered = movies_df[movies_df['year'] <= 1980].copy()
    
    return movies_filtered

In [29]:
# BERT embedding process 
@torch.no_grad()
def bert_embed(texts, max_len=256): 
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    ) # conversion
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]
    emb = F.normalize(cls, dim=-1)
    return emb.cpu().numpy().astype("float32") 

In [30]:
# Build text representation of movies
def build_movie_text(row):
    # include genre
    clean_genres = str(row['genres']).replace('|', ' ')
    
    # include year 
    year_str = ""
    if row['year'] is not None: 
        year_str = f"({int(row['year'])})"

    text = f"{row['clean_title']}. {year_str}. {clean_genres}"
    
    return text

In [31]:
# Offline step: create embeddings 
def run_offline_step(file_path, bucket_name, s3_key, local_pt):
    # check if embeddings have been made 
    try: 
        boto3.client('s3').head_object(Bucket=bucket_name, Key=s3_key)
        print("Embeddings have already been made.")
        if not os.path.exists(local_pt):
            print("Fetching embeddings from S3 bucket.")
            boto3.client('s3').download_file(bucket_name, s3_key, local_pt)

        output_data = torch.load(local_pt, weights_only=False)
        
        # cached embeddings: reconstruct movies_df with bert_text
        movies_filtered = filter_movies(file_path)
        movies_filtered['bert_text'] = movies_filtered.apply(build_movie_text, axis=1)
        return output_data, movies_filtered 
        
    except Exception as e: 
        print("Embeddings have not been made. Generating now.")

        # filter movies
        movies_filtered = filter_movies(file_path)
        
        # build bert text
        movies_filtered['bert_text'] = movies_filtered.apply(build_movie_text, axis=1)
        
        # generate embeddings using bert text
        item_vecs = bert_embed(movies_filtered['bert_text'].tolist())
        
        # upload embeddings to S3
        output_data = {
            'embeddings': item_vecs,
            'movie_ids': movies_filtered['movie_id'].values,
            'titles': movies_filtered['clean_title'].values,
            'years': movies_filtered['year'].values
        }
        torch.save(output_data, local_pt)
        boto3.client('s3').upload_file(local_pt, bucket_name, s3_key)
        
        print("Embeddings created and stored in S3")
        return output_data, movies_filtered

output_data, movies_df = run_offline_step("data/ml-1m/movies.dat", "dongjoo-26spring", "data/movie_embeddings_1980.pt", "movie_embeddings.pt")

Embeddings have already been made.


## Task 3: Recommend Movies

In [32]:
import pandas as pd

# Fetch ratings data
def load_ratings(file_path):
    columns = ['user_id', 'movie_id', 'rating', 'timestamp']
    
    ratings_df = pd.read_csv(
        file_path, 
        sep='::', 
        engine='python', 
        names=columns,
        encoding='latin-1'
    )
    
    return ratings_df

In [33]:
# Build user profile to generate personalized recommendaitnos 
def build_user_text(user_id, ratings_df, movies_df, N=10):
    # get user history 
    user_history = ratings_df[ratings_df["user_id"] == user_id]
    
    # sort movies by rating and timestamp to get top 10 
    hist = (user_history
            .sort_values(by=["rating", "timestamp"], ascending=False)
            .head(N)["movie_id"]
            .tolist())
    
    if not hist:
        return "no history", set()

    # get bert text of those movies 
    available_movies = movies_df[movies_df['movie_id'].isin(hist)]
    text_list = available_movies['bert_text'].tolist()
    
    if not text_list:
        return "no history", set(hist)
        
    return " ".join(text_list), set(hist)

In [34]:
import numpy as np

# Generate cold and top users
def get_cold_and_top_user(ratings_df):    
    # get top user
    interactions_per_user = ratings_df['user_id'].value_counts()
    top_5_percent = int(len(interactions_per_user) * 0.05)
    top_users = interactions_per_user.head(top_5_percent).index.tolist()
    random_top_user = np.random.choice(top_users)
    
    # get top user's summary
    top_user_history = ratings_df[ratings_df['user_id'] == random_top_user]
    last_interaction = top_user_history['timestamp'].max()
    
    # get/return cold user and top user
    return {
        'top_user': {
            'user_id': random_top_user,
            'last_interaction': last_interaction,
            'history': top_user_history
        },
        'cold_user': {
            'user_id': 'NEW_USER',
            'last_interaction': 'N/A',
            'history': pd.DataFrame()
        }
    }

In [35]:
# Generate recommendations for cold user 
def recommendations_for_cold_user(movie_data):
    titles = movie_data['titles']
    years = movie_data['years']
    
    # Since cold, just recommend 5 most recent movies 
    most_recent = years.argsort()[::-1][:5]
    
    recs = titles[most_recent].tolist()
    return recs

In [36]:
from scipy.spatial.distance import cdist

# Generate recommendations for top user 
def recommendations_for_top_user(user_id, ratings_df, movies_df, k=5):
    # build text and embed user history 
    user_text, seen = build_user_text(user_id, ratings_df, movies_df, N=10)
    
    if user_text == "no history":
        return recommendations_for_cold_user(movie_data)
        
    u = bert_embed([user_text]) 
    
    # find similar items
    # computes the distance between 'u' and every item in movie_data
    all_item_embeddings = movie_data['embeddings']
    distances = cdist(u, all_item_embeddings, metric='cosine')[0]
    
    # find most similar movies
    idx = np.argsort(distances)
    
    # get top 5 movies user hasn't seen yet
    recs = []
    for j in idx:
        iid = movie_data['movie_ids'][j]
        if iid not in seen:
            recs.append(movie_data['titles'][j])
        
        if len(recs) == k:
            break
            
    return recs

In [37]:
ratings = load_ratings("data/ml-1m/ratings.dat")
movie_data = output_data
movies_filtered = movies_df

# Get recommendations for cold and top users and call above functions 
def get_all_recommendations(movie_data, ratings_df, movies_df, bucket_name):
    user_context = get_cold_and_top_user(ratings_df)

    print("Generating recommendations for cold user.")
    cold_recs = recommendations_for_cold_user(movie_data)

    print("Generating recommendations for top user.")
    top_user_info = user_context['top_user']
    top_recs = recommendations_for_top_user(top_user_info['user_id'], ratings_df, movies_df, k=5)

    results = [
        {
            "User_Type": "Top User",
            "User_ID": str(top_user_info['user_id']),
            "Last_Interaction_Time": str(top_user_info['last_interaction']),
            "Num_Ratings": len(top_user_info['history']),
            "Avg_Rating": round(top_user_info['history']['rating'].mean(), 2),
            "Recommended_Movies": top_recs
        },
        {
            "User_Type": "Cold User",
            "User_ID": "NEW_USER",
            "Last_Interaction_Time": "N/A",
            "Num_Ratings": 0,
            "Avg_Rating": "N/A",
            "Recommended_Movies": cold_recs
        }
    ]
    
    return results

final_results = get_all_recommendations(movie_data, ratings, movies_filtered, "dongjoo-26spring")

Generating recommendations for cold user.
Generating recommendations for top user.


In [38]:
import json 

# Save and upload generated recommendations to S3 bucket
def save_recs_to_s3(results, bucket_name, s3_key):
    local_file = "task3_recommendations.json"
    with open(local_file, 'w') as f:
        json.dump(results, f, indent=4)
    
    boto3.client('s3').upload_file(local_file, bucket_name, s3_key)
    print("Uploaded recommendations to S3 bucket.")

save_recs_to_s3(final_results, "dongjoo-26spring", "results/task3_recs.json")

Uploaded recommendations to S3 bucket.


## Task 4: Repeat Tasks 2-3 With Full Dataset

In [39]:
import json 

# Get full movie dataset (unfiltered)
def load_all_movies(file_path):
    movies_df = pd.read_csv(
        file_path,
        sep='::',
        engine='python',
        encoding='latin-1',
        names=['movie_id', 'title', 'genres']
    )
    movies_df[['clean_title', 'year']] = movies_df.apply(get_year_and_fix_title, axis=1)
    return movies_df.copy()

# Generate recommendations for cold and top users using the full movie dataset
def get_all_recommendations_full(bucket_name, movies_path, ratings_path):
    # embeddings for all movies
    output_data_full, movies_df_full = run_offline_step(
        movies_path,
        bucket_name,
        "data/movie_embeddings_full.pt",
        "movie_embeddings_full.pt"
    )
    
    # get recommendations
    ratings_full = load_ratings(ratings_path)
    final_results_full = get_all_recommendations(output_data_full, ratings_full, movies_df_full, bucket_name)
    
    # save to S3
    save_recs_to_s3(final_results_full, bucket_name, "results/task4_recs.json")

    print("Recommendations for task 4 have been uploaded to S3.")
    return final_results_full

get_all_recommendations_full("dongjoo-26spring", "data/ml-1m/movies.dat", "data/ml-1m/ratings.dat")

Embeddings have already been made.
Generating recommendations for cold user.
Generating recommendations for top user.
Uploaded recommendations to S3 bucket.
Recommendations for task 4 have been uploaded to S3.


[{'User_Type': 'Top User',
  'User_ID': '5954',
  'Last_Interaction_Time': '957707693',
  'Num_Ratings': 1000,
  'Avg_Rating': 3.29,
  'Recommended_Movies': ['Babes in Toyland',
   'Blue Hawaii',
   'Anchors Aweigh',
   "Pot O' Gold",
   'Sunset Blvd. (a.k.a. Sunset Boulevard)']},
 {'User_Type': 'Cold User',
  'User_ID': 'NEW_USER',
  'Last_Interaction_Time': 'N/A',
  'Num_Ratings': 0,
  'Avg_Rating': 'N/A',
  'Recommended_Movies': ['Airplane!',
   'American Gigolo',
   'Howling, The',
   'Make Them Die Slowly (Cannibal Ferox)',
   'Private Benjamin']}]

## Task 5: User Profile

In [40]:
import json

# Create personal user profile and generate recommendations 
def get_personal_recommendations(bucket_name, movie_data, movies_df):
    # setting up my personal profile (rated romantic movies higher)
    my_ratings = [
        {"movie_id": 590, "title": "Snow White and the Seven Dwarfs", "rating": 5},
        {"movie_id": 3335, "title": "Titanic", "rating": 5},
        {"movie_id": 3345, "title": "Love Is a Many-Splendored Thing", "rating": 5},
        {"movie_id": 2577, "title": "House of Dracula", "rating": 2},
        {"movie_id": 2027, "title": "Sleeping Beauty", "rating": 5},
        {"movie_id": 907, "title": "Wizard of Oz, The", "rating": 1},
        {"movie_id": 2287, "title": "King Kong", "rating": 2},
        {"movie_id": 257, "title": "Star Wars: Episode IV - A New Hope", "rating": 1},
        {"movie_id": 2462, "title": "Battle for the Planet of the Apes", "rating": 1},
        {"movie_id": 2460, "title": "Planet of the Apes", "rating": 1},
    ]

    # save profile to S3
    profile = {"user_id": "my_profile", "ratings": my_ratings}
    with open("my_profile.json", "w") as f:
        json.dump(profile, f, indent=4)
    boto3.client('s3').upload_file("my_profile.json", bucket_name, "results/my_profile.json")
    print("Profile has been saved to S3.")

    # build user text 
    rated_ids = [r["movie_id"] for r in my_ratings]
    available = movies_df[movies_df["movie_id"].isin(rated_ids)]
    text_list = available["bert_text"].tolist()

    user_text = " ".join(text_list)
    u = bert_embed([user_text])
    distances = cdist(u, movie_data["embeddings"], metric="cosine")[0]
    idx = np.argsort(distances)
    seen = set(rated_ids)
    recs = []
    for j in idx:
        iid = movie_data["movie_ids"][j]
        if iid not in seen:
            recs.append(str(movie_data["titles"][j]))
        if len(recs) == 5:
            break

    # save recommendations to S3
    results = {"user_id": "my_profile", "recommended_movies": recs}
    with open("my_recs.json", "w") as f:
        json.dump(results, f, indent=4)
    boto3.client('s3').upload_file("my_recs.json", bucket_name, "results/my_recs.json")
    print("Recommendations have been saved to S3.")
    return results

get_personal_recommendations("dongjoo-26spring", output_data, movies_df)

Profile has been saved to S3.
Recommendations have been saved to S3.


{'user_id': 'my_profile',
 'recommended_movies': ['Willy Wonka and the Chocolate Factory',
  'Lady and the Tramp',
  'Mighty Joe Young',
  'Herbie Rides Again',
  'Faster Pussycat! Kill! Kill!']}